# RankExt Head-LR Small Probe -- Colab Runner

SMOKE / DIAGNOSTIC ONLY -- NOT THESIS BENCHMARK RESULTS.

Restores RankExt to its historical, pre-job4972616 (pre-task-scale-calibration)
pipeline (job4971615 source: `experiments_prepared/final_9method_5x20_performance_recovery.py`)
and sweeps only the RankExt head/classifier LR multiplier over a small,
conservative set (0.5, 1.0, 2.0) on a reduced 3-task / 2-epoch protocol,
testing only the `rank_extension_factor_orth_lam50_fullkd_T2_protect30`
(primary) and `rank_extension_factor_orth_lam50` (secondary) arms. SimpleAvg
and every other RankExt variant are off. See
`R7/rankext_headlr_small_probe_colab_readiness.md` for the full scientific-
equivalence audit against the cluster probe
(`experiments_prepared/rankext_headlr_small_probe.py`) this notebook runs
unmodified (only Colab-adaptation additions -- GPU-required check, batch-size
override -- differ; see that file's own "COLAB ADAPTATION BLOCK" comment).

## Cell 1 -- Runtime / GPU check

In [ ]:
!nvidia-smi


## Cell 2 -- Clone or update the repository

In [ ]:
import os

REPO_URL = "https://github.com/Kasra-Shp/vit-cifar100-project.git"
REPO_DIR = "/content/vit-cifar100-project"

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL} -> {REPO_DIR}")
    get_ipython().system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    print(f"{REPO_DIR} already exists -- pulling latest instead of re-cloning.")
    get_ipython().system(f'cd {REPO_DIR} && git pull --ff-only')

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


## Cell 3 -- Dependency inspection / install

Colab ships `torch`, `torchvision`, `numpy`, `pandas`, `matplotlib` pre-installed
with CUDA already wired -- this cell does NOT reinstall/downgrade any of those.
It only checks for and installs what the probe script actually imports and Colab
does *not* ship by default: `transformers`, `peft`, `datasets`. `scipy` is used
only for optional smooth-curve plotting (the script falls back to plain
polylines if missing) and is Colab stock anyway. `scikit-learn` is NOT imported
anywhere in this probe -- listed in the original spec as a general dependency to
check, but this script has no sklearn dependency, so it is intentionally
excluded from the install list below (checked but not required).

In [ ]:
import importlib

def check(pkg):
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg}: OK ({getattr(m, '__version__', 'version unknown')})")
        return True
    except ImportError:
        print(f"  {pkg}: MISSING")
        return False

print("--- Pre-installed Colab stock (never reinstalled here) ---")
for pkg in ["torch", "torchvision", "numpy", "pandas", "matplotlib", "scipy"]:
    check(pkg)

print()
print("--- Probe-specific dependencies (installed only if missing) ---")
need_install = []
for pkg, pipname in [("transformers", "transformers"), ("peft", "peft"), ("datasets", "datasets")]:
    if not check(pkg):
        need_install.append(pipname)

print()
print("--- Checked but NOT required by this probe (informational only) ---")
check("sklearn")  # not imported anywhere in rankext_headlr_small_probe_colab.py

if need_install:
    print("\nInstalling missing packages:", need_install)
    get_ipython().system(f"pip install -q {' '.join(need_install)}")
else:
    print("\nAll required packages already present -- nothing to install.")


## Cell 4 -- Optional Google Drive mount (persistent result backup)

Not required. Set `MOUNT_DRIVE = True` below to enable copying each completed
LR condition's result directory to
`/content/drive/MyDrive/vit-cifar100-project-results/` after it finishes
(see Cell 7's sweep driver and Cell 11).

In [ ]:
MOUNT_DRIVE = False  # set True to enable Drive backup

DRIVE_BACKUP_DIR = None
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_DIR = "/content/drive/MyDrive/vit-cifar100-project-results/"
    import os
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

print("Drive backup dir:", DRIVE_BACKUP_DIR if DRIVE_BACKUP_DIR else "(disabled)")


## Cell 5 -- Colab config / paths / batch size

`COLAB_BATCH_SIZE` overrides `BATCH_LORA` only (the batch size RankExt's
training path uses) -- nothing else. Default 8 is a conservative choice for a
16GB T4; the cluster ran `BATCH_LORA=16` on A40 (48GB). Raise this to 16 (or
higher) on an L4/A100 runtime if you want closer parity with the cluster run.
If you hit a CUDA OOM in Cell 7, lower this value and re-run -- do NOT change
any other hyperparameter to "fix" an OOM.

In [ ]:
import os

os.environ["COLAB_BATCH_SIZE"] = "8"  # edit this if you have more VRAM (e.g. "16" on L4/A100)
print("COLAB_BATCH_SIZE =", os.environ["COLAB_BATCH_SIZE"])

SCRIPT_PATH = "experiments_prepared/rankext_headlr_small_probe_colab.py"
LR_MULTIPLIERS = [0.5, 1.0, 2.0]  # historical baseline = 1.0; see readiness doc Section 5 for why no wider range
RESULTS_ROOT = "results"

print("SCRIPT_PATH:", SCRIPT_PATH)
print("LR_MULTIPLIERS:", LR_MULTIPLIERS)


## Cell 6 -- Static probe verification (no training)

Mirrors `R7/rankext_headlr_small_probe_colab_readiness.md`'s static checks:
compiles cleanly, no functional `/nfsd`/`sbatch`/`srun` dependency, no active
task-scale calibration, exactly the 2 intended RankExt methods (+ SimpleAvg
off), and the LR-sweep/task/epoch/seed constants match spec.

In [ ]:
import subprocess

print("--- py_compile ---")
r = subprocess.run(["python", "-m", "py_compile", SCRIPT_PATH], capture_output=True, text=True)
print("OK" if r.returncode == 0 else f"FAILED:\n{r.stdout}\n{r.stderr}")
assert r.returncode == 0, "Static compile check failed -- do not proceed."

def grep_count(pattern):
    r = subprocess.run(["grep", "-c", pattern, SCRIPT_PATH], capture_output=True, text=True)
    return int(r.stdout.strip() or "0")

print("\n--- Functional /nfsd or sbatch/srun dependency (expect: comment-only or zero) ---")
r = subprocess.run(["grep", "-n", "/nfsd\\|sbatch\\|srun", SCRIPT_PATH], capture_output=True, text=True)
functional_hits = [ln for ln in r.stdout.splitlines() if "#" not in ln.split(":", 1)[-1][:4]]
print(f"Total mentions: {len(r.stdout.splitlines())}; apparent non-comment hits: {len(functional_hits)}")

print("\n--- Task-scale calibration must be ABSENT (mentions here should only be COLAB ADAPTATION doc-comments) ---")
print("PerTaskScaleCalibration/RankExtCalibratedModel/apply_rankext_task_scale_calibration occurrences:",
      grep_count("PerTaskScaleCalibration\\|RankExtCalibratedModel\\|apply_rankext_task_scale_calibration"))

print("\n--- GPU-required check present ---")
print("torch.cuda.is_available() hard-fail present:", grep_count("torch.cuda.is_available") > 0)

print("\nStatic verification complete.")


## Cell 7 -- Run the full small LR sweep (combined 0.5 / 1.0 / 2.0)

Pattern A (spec Section 8): a Python driver loops over the 3 LR multipliers,
invoking the probe script as a **subprocess** per value (safer than
import-and-reset for a 9700+-line module-execution script) with
`RANKEXT_HEADLR_PROBE_MULTIPLIER` set in that subprocess's environment. Each
LR value writes to its own `RUN_NAME_BASE`-tagged output directory (the
script's own existing mechanism -- unchanged here), so runs never overwrite
each other.

Resume/skip (spec Section 9): before launching a given LR value, this driver
globs for an already-completed run of that LR (a
`{RUN_NAME_BASE}_*/reports/final_9method_output_checklist.txt` containing
`OVERALL PASS`) and skips straight to the next LR if found -- so re-running
this cell after a Colab disconnect does not repeat already-finished
conditions. This is directory-glob-based skip logic only, not a new
checkpoint framework.

In [ ]:
import glob
import os
import subprocess
import shutil

def run_name_base_for_lr(lr):
    tag = "lr" + str(float(lr)).replace(".", "p")
    return f"rankext_headlr_probe_3x20_2ep_{tag}_seed42"

def already_completed(lr):
    base = run_name_base_for_lr(lr)
    pattern = os.path.join(RESULTS_ROOT, f"{base}_*", "reports", "final_9method_output_checklist.txt")
    for path in glob.glob(pattern):
        with open(path, "r") as f:
            if "OVERALL PASS" in f.read():
                return path
    return None

completed_paths = {}
for lr in LR_MULTIPLIERS:
    print("=" * 80)
    existing = already_completed(lr)
    if existing:
        print(f"LR={lr}: already completed -- skipping ({existing})")
        completed_paths[lr] = existing
        continue

    print(f"LR={lr}: running rankext_headlr_small_probe_colab.py ...")
    env = os.environ.copy()
    env["RANKEXT_HEADLR_PROBE_MULTIPLIER"] = str(lr)
    proc = subprocess.run(["python", "-u", SCRIPT_PATH], env=env)
    if proc.returncode != 0:
        raise RuntimeError(f"LR={lr} run FAILED with exit code {proc.returncode} -- see output above.")

    existing = already_completed(lr)
    if not existing:
        raise RuntimeError(
            f"LR={lr} run finished but no completed-run marker was found under "
            f"{RESULTS_ROOT}/{run_name_base_for_lr(lr)}_*/reports/ -- check for a silent failure."
        )
    completed_paths[lr] = existing
    print(f"LR={lr}: completed -- {existing}")

    if DRIVE_BACKUP_DIR:
        run_dir = os.path.dirname(os.path.dirname(existing))  # .../reports/.. -> run dir
        dest = os.path.join(DRIVE_BACKUP_DIR, os.path.basename(run_dir))
        print(f"Backing up to Drive: {dest}")
        shutil.copytree(run_dir, dest, dirs_exist_ok=True)

print("=" * 80)
print("Sweep complete. Completed run marker paths:")
for lr, path in completed_paths.items():
    print(f"  LR={lr}: {path}")


## Cell 10 -- Collect / analyze results

Reads each LR condition's `final_9method_summary_table.csv` (the probe's
per-run summary table -- same schema/filename as the underlying canonical
script produces, just scoped to the 1-2 active RankExt arms) and produces
the 3 intended probe outputs:

- `R7/rankext_headlr_small_probe_report.md`
- `R7/rankext_headlr_small_probe_summary.csv`
- `R7/rankext_headlr_small_probe.png`

Per spec Section 11/14: labels every output SMOKE / DIAGNOSTIC ONLY. This is
a template analysis -- it reports what the summary tables and per-step
tables actually contain; it does NOT re-derive the stability-label judgment
calls (spec Section 11/12) automatically, since those require human
interpretation of the step-boundary CE/loss behavior, not just final
accuracy (spec Section 11: "Do NOT simply pick whichever LR gives the
highest tiny-smoke accuracy").

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for lr, checklist_path in completed_paths.items():
    run_dir = os.path.dirname(os.path.dirname(checklist_path))
    summary_csv = os.path.join(run_dir, "tables", "final_9method_summary_table.csv")
    if not os.path.exists(summary_csv):
        print(f"WARNING: no summary CSV found for LR={lr} at {summary_csv} -- skipping.")
        continue
    df = pd.read_csv(summary_csv)
    df["head_lr_multiplier"] = lr
    df["run_dir"] = run_dir
    rows.append(df)

if not rows:
    raise RuntimeError("No completed LR runs with a summary table found -- run Cell 7 first.")

all_df = pd.concat(rows, ignore_index=True)
summary_cols = [c for c in [
    "method", "head_lr_multiplier", "all_seen", "restricted_mean", "first_step",
    "later_steps_mean", "BWT", "forgetting", "run_dir",
] if c in all_df.columns]
summary_df = all_df[summary_cols].sort_values(["method", "head_lr_multiplier"])
print(summary_df.to_string(index=False))

os.makedirs("R7", exist_ok=True)
summary_df.to_csv("R7/rankext_headlr_small_probe_summary.csv", index=False)
print("\nWrote R7/rankext_headlr_small_probe_summary.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for method, sub in summary_df.groupby("method"):
    sub = sub.sort_values("head_lr_multiplier")
    axes[0].plot(sub["head_lr_multiplier"], sub["all_seen"], marker="o", label=method)
    if "restricted_mean" in sub.columns:
        axes[1].plot(sub["head_lr_multiplier"], sub["restricted_mean"], marker="o", label=method)
axes[0].set_title("Final open (all-seen) accuracy vs head-LR multiplier")
axes[0].set_xlabel("Head-LR multiplier")
axes[0].set_ylabel("All-seen accuracy (%)")
axes[1].set_title("Final restricted accuracy vs head-LR multiplier")
axes[1].set_xlabel("Head-LR multiplier")
axes[1].set_ylabel("Restricted accuracy (%)")
for ax in axes:
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
fig.suptitle("SMOKE / DIAGNOSTIC ONLY -- NOT THESIS BENCHMARK RESULTS", fontsize=9, color="crimson")
fig.tight_layout()
fig.savefig("R7/rankext_headlr_small_probe.png", dpi=200)
print("Wrote R7/rankext_headlr_small_probe.png")

with open("R7/rankext_headlr_small_probe_report.md", "w") as f:
    f.write("# RankExt Head-LR Small Probe -- Report\n\n")
    f.write("SMOKE / DIAGNOSTIC ONLY -- NOT THESIS BENCHMARK RESULTS.\n\n")
    f.write("Generated by Cell 10 of `rankext_headlr_small_probe_colab.ipynb`.\n\n")
    f.write("## Final metrics by LR multiplier\n\n")
    f.write(summary_df.to_markdown(index=False))
    f.write("\n\n## Interpretation\n\n")
    f.write(
        "This auto-generated section reports final numbers only. Per spec Section 11, "
        "do NOT declare a final LR decision from final accuracy alone -- inspect each "
        "run's `tables/final_9method_training_loss_history_by_epoch.csv` and "
        "`tables/final_9method_rankext_cl_retention_plasticity_trajectory.csv` for "
        "step-boundary CE-spike / retention behavior before assigning a "
        "PROMISING / NEUTRAL / UNSTABLE / UNDERTRAINED / INCONCLUSIVE label, "
        "per spec Sections 11-12.\n"
    )
print("\nWrote R7/rankext_headlr_small_probe_report.md")


## Cell 11 -- Save / copy artifacts

In [ ]:
import shutil

print("Local R7 probe outputs:")
for fn in ["rankext_headlr_small_probe_report.md", "rankext_headlr_small_probe_summary.csv", "rankext_headlr_small_probe.png"]:
    path = os.path.join("R7", fn)
    print(" ", path, "EXISTS" if os.path.exists(path) else "MISSING")

if DRIVE_BACKUP_DIR:
    dest = os.path.join(DRIVE_BACKUP_DIR, "R7_probe_outputs")
    os.makedirs(dest, exist_ok=True)
    for fn in ["rankext_headlr_small_probe_report.md", "rankext_headlr_small_probe_summary.csv", "rankext_headlr_small_probe.png"]:
        src = os.path.join("R7", fn)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(dest, fn))
    print(f"\nCopied R7 probe outputs to {dest}")
else:
    print("\nDrive backup disabled -- outputs remain only in the Colab VM's local filesystem "
          "(download them manually, e.g. via the Colab file browser, before the runtime recycles).")
